### **LLM Quantization**

In [ ]:
!pip install auto-gptq optimum accelerate transformers
!pip -q install -U huggingface_hub autoawq
!pip install -U pip setuptools wheel ninja
!pip install -U "torch" "torchvision" --index-url https://download.pytorch.org/whl/cu121
!pip install transformers torch --quiet
!pip install "auto-gptq>=0.7.1" --no-build-isolation
!nvidia-smi

PTQ (Post-Training Quantization)

In [ ]:
import os
import time
import torch
import json
import glob
import torch.nn as nn
import torch.nn.functional as F
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.quantization import quantize_dynamic
from auto_gptq import AutoGPTQForCausalLM, BaseQuantizeConfig
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments, AutoConfig
from huggingface_hub import notebook_login, whoami, upload_folder
from peft import LoraConfig, get_peft_model, TaskType
import torch.quantization as tq
from awq import AutoAWQForCausalLM
from transformers import DataCollatorForLanguageModeling
from transformers import BitsAndBytesConfig
from torch.ao.quantization import (
    QConfig,
    fake_quantize_per_tensor_affine,
    default_qat_qconfig,
    prepare_qat,
    convert,
)
import bitsandbytes as bnb
import matplotlib.pyplot as plt
import numpy as np
import torch.quantization
import warnings
warnings.filterwarnings('ignore')

X, y = make_moons(n_samples=1000, noise=0.1, random_state=42)
X, X.shape

In [ ]:
X, y = make_moons(n_samples=500, noise=0.2, random_state=42)
plt.scatter(X[:, 0], X[:, 1], c=y, cmap='bwr')
plt.title("make_moons Dataset")
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.show()

In [ ]:
class BigMLP(nn.Module):
  def __init__(self):
    super().__init__()
    self.fc1 = nn.Linear(2, 128)
    self.fc2 = nn.Linear(128, 64)
    self.fc3 = nn.Linear(64, 64)
    self.fc4 = nn.Linear(64, 32)
    self.fc5 = nn.Linear(32, 16)
    self.fc6 = nn.Linear(16, 8)
    self.fc7 = nn.Linear(8, 1)

  def forward(self, x):
    x = F.relu(self.fc1(x))
    x = F.relu(self.fc2(x))
    x = F.relu(self.fc3(x))
    x = F.relu(self.fc4(x))
    x = F.relu(self.fc5(x))
    x = F.relu(self.fc6(x))
    return torch.sigmoid(self.fc7(x))

model_fp32 = BigMLP()
model_fp32.parameters()
optimizer = torch.optim.Adam(model_fp32.parameters(), lr=0.01)
loss_fn = nn.BCELoss()

for epoch in range(1000):
  model_fp32.train()
  optimizer.zero_grad()
  out = model_fp32(X_train)
  loss = loss_fn(out, y_train)
  loss.backward()
  optimizer.step()
  if epoch % 500 == 0:
    print(f"Epoch {epoch} | Loss: {loss.item():.4f}")

In [ ]:
def accuracy(model, x, y):
  model.eval()
  with torch.no_grad():
    preds = model(x)
    preds = (preds > 0.5).float()
    return (preds == y).float().mean().item()

model_int8 = quantize_dynamic(
    model_fp32,
    {nn.Linear},
    dtype=torch.qint8
)

print(model_int8)
print("\nFP32 Accuracy:", accuracy(model_fp32, X_test, y_test))

In [ ]:
model_fp32.state_dict()
model_int8.state_dict()

In [ ]:
torch.save(model_fp32.state_dict(), "model_fp32.pt")
torch.save(model_int8.state_dict(), "model_dynamic_int8.pt")


print("FP32 model size (MB):", os.path.getsize("model_fp32.pt") / 1e6)
print("INT8 model size (MB):", os.path.getsize("model_dynamic_int8.pt") / 1e6)

In [ ]:
def quantize_tensor(t, num_bits=8):
  qmin = -2**(num_bits-1)
  qmax = 2**(num_bits-1) - 1
  min_val, max_val = t.min(), t.max()
  scale = (max_val - min_val) / float(qmax - qmin + 1e-8)
  zp = torch.round(-min_val / scale).to(torch.int32)
  q_t = torch.clamp(torch.round(t / scale) + zp, qmin, qmax).to(torch.int8)
  return q_t, scale, zp

def dequantize_tensor(q_t, scale, zp):
  return (q_t.float() - zp) * scale

new_model = BigMLP()
for name, param in new_model.named_parameters():
  print(name)
  print(param.shape)

In [ ]:
with torch.no_grad():
  for (name_fp, param_fp), (name_q, param_q) in zip(model_fp32.named_parameters(), new_model.named_parameters()):
    q_param, scale, zp = quantize_tensor(param_fp.data)
    dq_param = dequantize_tensor(q_param, scale, zp)
    param_q.data.copy_(dq_param)

print("\n INT8 Accuracy:", accuracy(new_model, X_test, y_test))

In [ ]:
X, y = make_moons(n_samples=1000, noise=0.1, random_state=42)
X = StandardScaler().fit_transform(X)
X = torch.tensor(X, dtype=torch.float32)
y = torch.tensor(y.reshape(-1, 1), dtype=torch.float32)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

class MLP(nn.Module):
  def __init__(self):
    super().__init__()
    self.fc1 = nn.Linear(2, 16)
    self.fc2 = nn.Linear(16, 8)
    self.fc3 = nn.Linear(8, 1)

  def forward(self, x):
    x = F.relu(self.fc1(x))
    x = F.relu(self.fc2(x))
    x = torch.sigmoid(self.fc3(x))
    return x

model_fp32 = MLP()
optimizer = torch.optim.Adam(model_fp32.parameters(), lr=0.01)
loss_fn = nn.BCELoss()

for epoch in range(2000):
  model_fp32.train()
  optimizer.zero_grad()
  out = model_fp32(X_train)
  loss = loss_fn(out, y_train)
  loss.backward()
  optimizer.step()

def accuracy(model, X, y):
  model.eval()
  with torch.no_grad():
    preds = model(X)
    preds = (preds > 0.5).float()
    acc = (preds == y).float().mean().item()
    return acc

print("\nFP32 Accuracy:", accuracy(model_fp32, X_test, y_test))

In [ ]:
def quantize_tensor(t, num_bits=8):
    qmin = -2**(num_bits - 1)
    qmax = 2**(num_bits - 1) - 1
    min_val, max_val = t.min(), t.max()
    scale = (max_val - min_val) / (qmax - qmin + 1e-8)
    zp = torch.round(-min_val / scale).to(torch.int32)
    q_t = torch.clamp(torch.round(t / scale) + zp, qmin, qmax).to(torch.int8)
    return q_t, scale, zp

def dequantize_tensor(q_t, scale, zp):
    return (q_t.float() - zp) * scale

def get_activation_stats(model, X_sample):
  act_ranges = {}
  hooks = []

  def register_hook(module):
    def hook_fn(module, input, output):
      act_min = output.min().item()
      act_max = output.max().item()
      act_ranges[name] = (act_min, act_max)

    return hook_fn

  for name, module in model.named_modules():
    if isinstance(module, nn.Linear):
      hooks.append(module.register_forward_hook(register_hook(name)))

  with torch.no_grad():
    model.eval()
    model(X_sample)

  for hook in hooks:
    hook.remove()

  return act_ranges

X_calib = X_train[:100]
act_ranges = get_activation_stats(model_fp32, X_calib)
act_ranges

In [ ]:
class QuantizedMLP(nn.Module):
  def __init__(self, fp_model, act_ranges):
    super().__init__()
    self.fc1 = nn.Linear(2, 16)
    self.fc2 = nn.Linear(16, 8)
    self.fc3 = nn.Linear(8, 1)

    with torch.no_grad():
      for (name_fp, param_fp), (name_q, param_q) in zip(fp_model.named_parameters(), self.named_parameters()):
        q_param, scale, zp = quantize_tensor(param_fp.data)
        dq_param = dequantize_tensor(q_param, scale, zp)
        param_q.data.copy_(dq_param)

    self.act_scales = {}
    for name, (amin, amax) in act_ranges.items():
      scale = (amax - amin) / 255.0
      zp = round(-amin / scale) if scale > 0 else 0
      self.act_scales[name] = (scale, zp)

  def quant_act(self, x, layer_name):
    scale, zp = self.act_scales[layer_name]
    q_x = torch.clamp(torch.round(x / scale) + zp, 0, 255).to(torch.int8)
    dp_x = (q_x.float() - zp) * scale
    return dp_x

  def forward(self, x):
    x = F.relu(self.quant_act(self.fc1(x), 'fc1'))
    x = F.relu(self.quant_act(self.fc2(x), 'fc2'))
    x = torch.sigmoid(self.fc3(x))
    return x

model_static_ptq = QuantizedMLP(model_fp32, act_ranges)
print("\nStatic INT8 Accuracy:", accuracy(model_static_ptq, X_test, y_test))

QAT (Quantization Aware Training)

In [ ]:
X, y = make_moons(n_samples=500, noise=0.2, random_state=42)
X = StandardScaler().fit_transform(X)
X = torch.tensor(X, dtype=torch.float32)
y = torch.tensor(y.reshape(-1, 1), dtype=torch.float32)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

def accuracy(model, X, y):
  model.eval()
  with torch.no_grad():
    preds = model(X)
    preds = (preds > 0.5).float()
    return (preds == y).float().mean().item()

class QATMLP(nn.Module):
  def __init__(self):
    super().__init__()
    self.quant = torch.quantization.QuantStub()
    self.dequant = torch.quantization.DeQuantStub()
    self.fc1 = nn.Linear(2, 16)
    self.fc2 = nn.Linear(16, 8)
    self.fc3 = nn.Linear(8, 1)

  def forward(self, x):
    x = self.quant(x)
    x = F.relu(self.fc1(x))
    x = F.relu(self.fc2(x))
    x = torch.sigmoid(self.fc3(x))
    x = self.dequant(x)
    return x

model = QATMLP()
model.qconfig = torch.quantization.get_default_qat_qconfig('fbgemm')
torch.quantization.prepare_qat(model, inplace=True)

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
loss_fn = nn.BCELoss()

for epoch in range(2000):
    model.train()
    optimizer.zero_grad()
    out = model(X_train)
    loss = loss_fn(out, y_train)
    loss.backward()
    optimizer.step()
    if epoch % 500 == 0:
        print(f"Epoch {epoch} | Loss: {loss.item():.4f}")

model_int8 = torch.quantization.convert(model.eval(), inplace=False)
print("Accuracy before convert():", accuracy(model, X_test, y_test))
print("QAT INT8 Accuracy:", accuracy(model_int8, X_test, y_test))

### **Gradient Post-Training Quantization(GPTO)**

In [ ]:
model_id = "TheBloke/Llama-2-7B-Chat-GPTQ"
tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=True)
model = AutoGPTQForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    use_safetensors=True,
    trust_remote_code=True,
    use_triton=False,
)

prompt = "Explain the difference between GPTQ and AWQ in simple terms."
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
outputs = model.generate(
    **inputs,
    max_new_tokens=1024,
    do_sample=True,
    top_p=0.9,
    temperature=0.7,
)

response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(response)

In [ ]:
model_id = "tiiuae/falcon-rw-1b"
quant_config = BaseQuantizeConfig(
    bits=4,
    group_size=128,
    desc_act=False,
)

tokenizer = AutoTokenizer.from_pretrained(
    model_id,
    use_fast=True,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoGPTQForCausalLM.from_pretrained(
    model_id,
    quantize="gptq",
    trust_remote_code=True,
)

example_texts = [
    "What is AI?",
    "Explain the purpose of Falcon model.",
    "List 3 benefits of GPTQ.",
    "How does quantization work in LLMs?",
    "Give an example of a large language model.",
]

examples = tokenizer(example_texts, return_tensors="pt", padding=True, truncation=True)
examples = [ {k: v[i] for k, v in examples.items()} for i in range(len(example_texts)) ]

In [ ]:
output_dir = "falcon-rw-1b-gptq"
model.save_quantized(output_dir, use_safetensors=True)
tokenizer.save_pretrained(output_dir)
AutoConfig.from_pretrained(model_id, trust_remote_code=True).save_pretrained(output_dir)

In [ ]:
model_path = "falcon-rw-1b-gptq"
tokenizer = AutoTokenizer.from_pretrained(model_path, use_fast=True)
model = AutoGPTQForCausalLM.from_pretrained(
    model_path,
    device_map="auto",
    trust_remote_code=False,
    use_safetensors=True,
    use_triton=False,
    disable_exllamav2=True,
)

prompts = [
    "What is the capital of India?",
    "Explain how quantization improves inference.",
    "What is the Falcon model used for?",
    "List 3 applications of LLMs.",
    "Define GPTQ in simple terms.",
]

total_time=0.0
all_outputs = []

for i, prompt in enumerate(prompts):
  inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
  start = time.time()

  outputs = model.generate(
      **inputs,
      max_new_tokens=64,
      do_sample=False,
  )

  inference_time = time.time() - start
  total_time += inference_time
  response = tokenizer.decode(outputs[0], skip_special_tokens=True)
  all_outputs.append((prompt, response, round(inference_time, 3)))

for i, (prompt, response, t) in enumerate(all_outputs):
    print(f"\n Prompt {i+1}: {prompt}")
    print(f" Response: {response}")
    print(f" Inference Time: {t} seconds")

avg_time = total_time / len(prompts)
print(f"\n Average Inference Time: {round(avg_time, 3)} seconds per prompt")

In [ ]:
notebook_login()
whoami()

In [ ]:
upload_folder(
    repo_id="Emon3469/quantized-falcon-rw-1b",
    folder_path="./falcon-rw-1b-gptq",
    commit_message="Upload GPTQ quantized Falcon model"
)

In [ ]:
print(os.listdir(path))
with open(f"{path}/quantize_config.json") as f:
    print(json.load(f))

print(glob.glob(f"{path}/gptq_model-*.safetensors"))

### **Activation-aware Weight Quantization (AWQ)**

In [ ]:
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_USE_SDPA"] = "0"
base_model = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU: {torch.cuda.get_device_name()}")

if torch.cuda.is_available():
    torch.cuda.empty_cache()
gc.collect()

In [ ]:
tok = AutoTokenizer.from_pretrained(
    base_model,
    use_fast=True,
    trust_remote_code=True,
)

if tok.pad_token is None:
  tok.pad_token = tok.eos_token

model = AutoAWGForCausalLM.from_pretrained(
    base_model,
    low_cpu_mem_usage=True,
    use_cache=False,
    trust_remote_code=True,
    torch_dtype=torch.float16,
    device_map={"": 0} if torch.cuda.is_available() else "cpu",
)

model.eval()

In [ ]:
quant_config = {
    "zero_point" : True,
    "q_group_size" : 128,
    "w_bit" : 4,
    "version" : "GEMM"
}

calib_texts = [
    "The quick brown fox jumps over the lazy dog.",
    "Machine learning models process data efficiently.",
    "Natural language understanding is advancing rapidly.",
    "Deep neural networks learn complex patterns.",
    "Artificial intelligence transforms technology.",
    "Computer vision recognizes objects accurately.",
    "Robotics integrates sensors and actuators.",
    "Algorithm optimization improves performance significantly.",
    "Data science extracts meaningful insights.",
    "Software engineering creates reliable systems."
] * 10

try:
  model.quantize(
      tok,
      quant_config=quant_config,
      calib_data=calib_texts,
      max_calib_seq_len=128,
      max_calib_samples=50,
      n_parallel_calib_samples=1
  )

  out_dir = "tinyllama-1.1b-awq"
  model.save_quantized(out_dir, safetensors=True)
  tok.save_pretrained(out_dir)

  print(f"Saving model to {out_dir}...")
  print(f"Model successfully quantized and saved to {out_dir}")

except Exception as e:
  print(f"Quantization failed with error: {str(e)}")
  del model
  if torch.cuda.is_available():
    torch.cuda.empty_cache()
  gc.collect()

  try:
    model = AutoAWQForCausalLM.from_pretrained(
        base_model,
        low_cpu_mem_usage=True,
        trust_remote_code=True,
        use_cache=False,
    )

    if torch.cuda.is_available():
      model = model.cuda()

    model.eval()

    model.quantize(
        tok,
        quant_config=quant_config,
        calib_data=calib_texts[:20],
        max_calib_seq_len=64,
        max_calib_samples=20,
        n_parallel_calib_samples=1
    )

    out_dir = "tinyllama-1.1b-awq"
    model.save_quantized(out_dir, safetensors=True)
    tok.save_pretrained(out_dir)
    config = AutoConfig.from_pretrained(base_model, trust_remote_code=True)
    config.save_pretrained(out_dir)

    print(f"Model successfully quantized with alternative approach and saved to {out_dir}")

  except Exception as e2:
    print(f"Alternative approach also failed: {str(e2)}")
    print("\nConsider using a pre-quantized model instead:")
    print("TheBloke/TinyLlama-1.1B-Chat-v1.0-AWQ")

finally:
  if torch.cuda.is_available():
    torch.cuda.empty_cache()
  gc.collect()

In [ ]:
cfg = AutoConfig.from_pretrained(
    base_model,
    trust_remote_code=True,
)
cfg.attn_implementation = "eager"

calib_tokens = [
    tok(
        text,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=128,
    ).input_ids
    for text in calib_texts[:50]
]

model.quantize(
    tok,
    quant_config=quant_config,
    calib_data=calib_texts,
    max_calib_seq_len=128,
    max_calib_samples=50,
    n_parallel_calib_samples=1,
)

In [ ]:
calib_tokens = [
    tok(
        text,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=128,
    ).input_ids[0].tolist()
    for text in calib_texts[:50]
]

model.quantize(
    tok,
    quant_config=quant_config,
    calib_data=calib_texts,
    max_calib_seq_len=128,
    max_calib_samples=50,
    n_parallel_calib_samples=1,
)

In [ ]:
model.quantize(
    tok,
    quant_config=quant_config,
    calib_data=calib_texts[:50],
    max_calib_seq_len=128,
    max_calib_samples=50,
    n_parallel_calib_samples=1,
)

In [ ]:
out_dir = "tinyllama-1.1b-awq"
model.save_quantized(out_dir, safetensors=True)
tok.save_pretrained(out_dir)
config = AutoConfig.from_pretrained(base_model, trust_remote_code=True)
config.save_pretrained(out_dir)

print(f"Saving model to {out_dir}...")

In [ ]:
try:
  model.quantize(
      tok,
      quant_config=quant_config,
      calib_data=calib_texts,
      max_calib_seq_len=128,
      max_calib_samples=50,
      n_parallel_calib_samples=1
  )

  out_dir = "tinyllama-1.1b-awq"
  model.save_quantized(out_dir, safetensors=True)
  tok.save_pretrained(out_dir)
  config = AutoConfig.from_pretrained(base_model, trust_remote_code=True)
  config.save_pretrained(out_dir)

  print(f"Saving model to {out_dir}...")
  print(f"Model successfully quantized and saved to {out_dir}")

except Exception as e:
  print(f"Quantization failed with error: {str(e)}")
  del model
  if torch.cuda.is_available():
    torch.cuda.empty_cache()
  gc.collect()

  try:
    model = AutoAWQForCausalLM.from_pretrained(
        base_model,
        low_cpu_mem_usage=True,
        trust_remote_code=True,
        use_cache=False,
    )

    if torch.cuda.is_available():
      model = model.cuda()

    model.eval()
    model.quantize(
        tok,
        quant_config=quant_config,
        calib_data=calib_texts[:20],
        max_calib_seq_len=64,
        max_calib_samples=20,
        n_parallel_calib_samples=1
    )

    out_dir = "tinyllama-1.1b-awq"
    mdl.save_quantized(out_dir, safetensors=True)
    tok.save_pretrained(out_dir)
    config = AutoConfig.from_pretrained(base_model, trust_remote_code=True)
    config.save_pretrained(out_dir)
    print(f"Model successfully quantized with alternative approach and saved to {out_dir}")

  except Exception as e2:
    print(f"Alternative approach also failed: {str(e2)}")

finally:
  if torch.cuda.is_available():
    torch.cuda.empty_cache()
  gc.collect()

Quantization-Aware Training(QAT)

In [ ]:
model_name = "distilgpt2"
model = AutoModelForCausalLM.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)

qat_config = tq.QConfig(
    activation=tq.MovingAverageMinMaxObserver.with_args(
        observer=tq.MovingAverageMinMaxObserver,
        quant_min=0,
        quant_max=255,
        dtype=torch.quint8,
        qscheme=torch.per_tensor_affine,
    ),
    weight=tq.MovingAverageMinMaxObserver.with_args(
        observer=tq.MinMaxObserver,
        quant_min=-128,
        quant_max=127,
        dtype=torch.qint8,
        qscheme=torch.per_tensor_symmetric,
    )
)

for name, module in model.named_modules():
  if isinstance(module, nn.Embedding):
    module.qconfig = None

model.qconfig = qat_config
model.train()

In [ ]:
model = model.to("cpu")
tq.prepare_qat(model, inplace=True)

inputs = tokenizer("Quantization Aware Training on LLMs!", return_tensors="pt")
inputs = {k: v.to("cpu") for k, v in inputs.items()}
labels = inputs["input_ids"]

optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)

for step in range(50):
  outputs = model(
      **inputs,
      labels=labels
  )
  loss = outputs.loss
  loss.backward()
  optimizer.step()
  optimizer.zero_grad()
  if step % 10 == 0:
    print(f"Step {step} | Loss: {loss.item()}")

model.eval()
qat_model = tq.convert(model, inplace=False)

In [ ]:
from datasets import Dataset

def qat_with_bitsandbytes():
  try:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16
    )

    model_name = "distilgpt2"
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=bnb_config,
        device_map="auto",
        torch_dtype=torch.float16,
    )
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    if tokenizer.pad_token is None:
      tokenizer.pad_token = tokenizer.eos_token

    model.gradient_checkpointing_enable()
    model.enable_input_require_grads()

    texts = [
        "Quantization Aware Training helps reduce model size while maintaining performance.",
        "Large Language Models can be efficiently quantized using various techniques.",
        "QAT allows models to adapt to quantization during training process."
    ]

    def tokenize_function(examples):
        return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=128)

    dataset = Dataset.from_dict({"text": texts})
    tokenized_dataset = dataset.map(tokenize_function, batched=True)

    training_args = TrainingArguments(
        output_dir="./qat_model",
        overwrite_output_dir=True,
        num_train_epochs=3,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        logging_steps=1,
        learning_rate=5e-5,
        fp16=True,
        save_strategy="no",
    )

    data_collator = DataCollatorForLanguageModeling(
        tokenizer=tokenizer,
        mlm=False,
    )
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_dataset,
        data_collator=data_collator,
    )

    trainer.train()
    test_input = "Quantization makes models"
    inputs = tokenizer(test_input, return_tensors="pt")

    with torch.no_grad():
      outputs = model.generate(
          **inputs,
          max_new_tokens=64,
          do_sample=False,
          temperature=0.7,
      )
      generated_text = tokenizer.decode(
          outputs[0],
          skip_special_tokens=True,
      )
      print(f"Generated text: {generated_text}")

    return model, tokenizer

  except ImportError:
    print("BitsAndBytes not available. Install with: pip install bitsandbytes")
    return None, None


def custom_qat_llm():
  class QuantizedLinear(nn.Module):
    def __init__(self, in_features, out_features, bias=True):
      super().__init__()
      self.in_features = in_features
      self.out_features = out_features
      self.weight_scale = nn.Parameter(torch.ones(1))
      self.weight_zero_point = nn.Parameter(torch.zeros(1))
      self.input_scale = nn.Parameter(torch.ones(1))
      self.input_zero_point = nn.Parameter(torch.zeros(1))
      self.weight = nn.Parameter(torch.randn(out_features, in_features))
      if bias:
        self.bias = nn.Parameter(torch.zeros(out_features))
      else:
        self.register_parameter("bias", None)

    def quantize_tensor(self, tensor, scale, zero_point):
      quantized = torch.round(tensor / scale + zero_point)
      quantized = torch.clamp(quantized, -128, 127)
      dequantized = (quantized - zero_point) * scale
      return dequantized

    def forward(self, input):
      quantized_weight = self.quantize_tensor(self.weight, self.weight_scale, self.weight_zero_point)
      quantized_input = self.quantize_tensor(input, self.input_scale, self.input_zero_point)
      return F.linear(quantized_input, quantized_weight, self.bias)

  def replace_linear_layers(model):
    for name, module in model.named_modules():
      if isinstance(module, nn.Linear):
        quant_layer = QuantizedLinear(module.in_features, module.out_features, module.bias is not None)
        quant_layer.weight.data = module.weight.data.clone()
        if module.bias is not None:
          quant_layer.bias.data = module.bias.data.clone()
        setattr(model, name, quant_layer)
      else:
        replace_linear_layers(module)

  model_name = "distilgpt2"
  model = AutoModelForCausalLM.from_pretrained(model_name)
  tokenizer = AutoTokenizer.from_pretrained(model_name)

  if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

  replace_linear_layers(model)
  optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)
  model.train()

  training_texts = [
      "Quantization Aware Training on LLMs reduces model size significantly.",
      "Custom QAT implementations allow fine control over quantization process.",
      "Fake quantization during training helps models adapt to quantization noise."
  ]

  for epoch in range(3):
    total_loss = 0
    for i, text in enumerate(training_texts):
      inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=128)
      labels = inputs["input_ids"].clone()
      outputs = model(**inputs, labels=labels)
      loss = outputs.loss
      loss.backward()
      optimizer.step()
      optimizer.zero_grad()
      total_loss += loss.item()

    avg_loss = total_loss / len(training_texts)
    print(f"Epoch {epoch + 1} | Average Loss: {avg_loss:.4f}")

  model.eval()
  test_input = "Quantization helps"
  inputs = tokenizer(test_input, return_tensors="pt")

  with torch.no_grad():
      outputs = model.generate(**inputs, max_new_tokens=15, do_sample=True, temperature=0.7)
      generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
      print(f"Generated text: {generated_text}")

  return model, tokenizer

def qat_with_peft():
  try:
    model_name = "distilgpt2"
    model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float16)
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    lora_config = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        inference_mode=False,
        r=16,
        lora_alpha=32,
        lora_dropout=0.1,
        target_modules=["c_attn", "c_proj"]
    )

    model = get_peft_model(model, lora_config)
    print(f"Trainable parameters: {model.print_trainable_parameters()}")
    optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)
    model.train()

    training_texts = [
        "PEFT with LoRA enables efficient quantization-aware training.",
        "QLoRA combines quantization with low-rank adaptation effectively.",
        "Parameter-efficient methods reduce memory usage during QAT."
    ]

    for epoch in range(2):
      total_loss = 0
      for text in training_texts:
        inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=128)
        labels = inputs["input_ids"].clone()
        outputs = model(**inputs, labels=labels)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        total_loss += loss.item()

      avg_loss = total_loss / len(training_texts)
      print(f"Epoch {epoch + 1} | Average Loss: {avg_loss:.4f}")

    model.eval()
    test_input = "PEFT enables"
    inputs = tokenizer(test_input, return_tensors="pt")

    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=15, do_sample=True, temperature=0.7)
        generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
        print(f"Generated text: {generated_text}")

    return model, tokenizer

  except ImportError:
    print("PEFT not available. Install with: pip install peft")
    return None, None

In [ ]:
print("=== Quantization-Aware Training for LLMs ===\n")
print("Method 1: BitsAndBytes QAT")
model1, tokenizer1 = qat_with_bitsandbytes()
if model1 is None:
    print("\nMethod 2: Custom QAT Implementation")
    model2, tokenizer2 = custom_qat_llm()
    print("\nMethod 3: PEFT with QLoRA")
    model3, tokenizer3 = qat_with_peft()

In [ ]:
model_name = "distilgpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)
model.qconfig = default_qat_qconfig

model.train()
qat_model = prepare_qat(model)

text = "Quantization aware training is powerful."
inputs = tokenizer(text, return_tensors="pt", padding=True)
optimizer = torch.optim.Adam(qat_model.parameters(), lr=1e-4)

for epoch in range(3):
    outputs = qat_model(**inputs, labels=inputs["input_ids"])
    loss = outputs.loss
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()
    print(f"Epoch {epoch+1} | Loss: {loss.item():.4f}")

qat_model.eval()
quantized_model = convert(qat_model)

with torch.no_grad():
    output = quantized_model.generate(**inputs, max_new_tokens=10)
    print("\nGenerated:", tokenizer.decode(output[0]))